# Nombre d'observations valides
<br/>
Dans ce notebook, nous allons reprendre les images utilisés dans le notebook de génération des masques de neige et calculer pour chaque pixels le nombre de fois où il a eu une valeur valide (autre que nodata).<br/>
Pour cela, nous allons faire une réduction temporelle et utiliser le process *count*.<br/>
<br/>
Import et installation des librairies nécessaires

In [ ]:
try:
    import openeo
except:
    !pip install openeo
    import openeo

In [ ]:
# Connexion à l'aggrégateur
con = openeo.connect("https://aggregator.geodes-lab.fr")
con.authenticate_oidc() 

## Création du datacube et du process graph

**Etape 1 :** Utiliser le process "load_collection" pour charger les données.

In [ ]:
df = con.load_collection(
    "S2-L2A-COG", 
    temporal_extent=["2022-01-01","2022-08-01"],
    bands=["B3", "B11"],
    max_cloud_cover=10,
    properties=[openeo.collection_property("tilename") == "T31TCH"]    
)
df

**Etape 2 :** Comptage des points valides

In [ ]:
df_valid = df.apply(lambda x: x.is_valid())

df_n_obs = df_valid.reduce_dimension(
    dimension="t",
    reducer="sum"
)
df_n_obs

## Création et lancement du job

In [ ]:
# On génère un GTiff en sortie
job_df = df_n_obs.create_job(title="Observations", out_format="GTiff")

In [ ]:
job_df.start_and_wait()  # environ 7 min

**Une fois que le job est terminé (status=finished), on récupère les résultats**

In [ ]:
# Aperçu des résultats
results = job_df.get_results()
results

In [ ]:
# Téléchargement
result_name = results.get_assets()[0].key
results.download_file(target="./results_snow/nobs.tif", key=result_name)

## Visualisation des résultats

In [ ]:
import rasterio
import matplotlib.pyplot as plt

In [ ]:
with rasterio.open("results_snow/nobs.tif") as src:
    data = src.read(1)

plt.figure()
plt.imshow(data)
plt.colorbar()
plt.axis("off")
plt.show()

In [ ]:
# Clean

import shutil

shutil.rmtree("./results_snow/")